# v120_night_build — the night build: every planned switch on, one run

| Field | Value |
|---|---|
| **Version** | `v120_night_build` |
| **Plan group** | INT (integrated build of the night-build plan, TRACKER #63–#73) |
| **Parent version** | v110 (`v110_m3_features`) |
| **Author** | M1 rajaguru2004 (integration; M2, M3 feature branches) |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

One integrated run instead of one test run per idea (TRACKER "Night build"): the parent's
pipeline (normalisation rules v5, dense blocking, two-stage matcher, tight-mock rule) with
every item of the night-build plan switched on:

* stage-1 groups +['nofill']
* stage-2 extra groups ['tok_evidence']
* blocking nofill_max_group=50
* normalisation learn_fillers=True
* token evidence learn=True
* stage 1 in 2 bags
* stage 2 on ['fit', 'tune'], 127 leaves, 3 seed(s)

Normalisation rules v5 (French N° marker, compagnie, bis/crs/psg/appt, `+`/`et` -> and, leet legal forms, the record's own country word out of the core name) are the code default from this version on. M2's phonetic group was not rebased in time and is not in this build.

## 1. Hypothesis

* **Change vs parent (v110):** the switches above, together; the mock fold, the filter and the
  tight-mock tuning are the parent's.
* **Why:** each item targets a measured loss of the parent: true pairs never blocked, true pairs
  filtered or below the rule, and the French and alias spellings the pool generator uses.
* **Check:** est_public on the mock val entities against the parent's; then the upload.
* **Discard if:** est_public does not beat the parent by more than 0.001.

## 2. Setup

The parent's configuration with the switches of the spec applied (printed below with the
blocking key), stage 1 on the GPU in bags, stage 2 as specified.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import TopKSpec
from entity_resolution.data import isin
from entity_resolution.decision import DecisionRule, apply_rule, decide, tune_expected
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.mock import FP_WEIGHT, PUBLIC_OFFSET, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.evidence import EvidenceConfig
from entity_resolution.pipeline import (
    Fitted, PipelineConfig, learn_fillers, learn_token_evidence, learn_token_map, mock_scores,
    peak_rss_gb, tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.stage1 import absent_from_mock, clean, fit_stage1, write_chunks
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    SeedMean, TwoStage, TwoStageConfig, fit_stage2, mock_scored, mock_stage1,
    run_test_two_stage,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v120_night_build"
ARTIFACTS = EXP_DIR / "artifacts"
M3_GROUPS = ("idf", "token_freq", "ctx_idf", "address_extra")    # PR #8 (M3), as in v040
# B6 (v105/v106): bigger word top-k and caps, sim-first cap; v109 adds the name + address-number
# exact pass (groups of at most 100) and a cap of 120 candidates per S1 entity
B6 = replace(PipelineConfig().blocking,
             name_addr_word=replace(PipelineConfig().blocking.name_addr_word, top_k=50),
             max_per_s1=100, exact_max_group=200,
             addr_char=TopKSpec("addr_norm", "word", (1, 2), top_k=10, min_sim=0.3,
                                max_df=0.01, max_df_abs=10_000),
             cap_order="sim_first")
assert replace(B6, name_num_max_group=100, max_per_s1=120).key() == "db1f33c4"   # v110's
NIGHT_GROUPS = ('nofill',)      # stage-1 groups added tonight
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency", *M3_GROUPS, *NIGHT_GROUPS),
                     model=MatcherParams(n_estimators=4000),
                     normalise=replace(PipelineConfig().normalise, learn_fillers=True),
                     evidence=EvidenceConfig(learn=True),
                     blocking=replace(B6, name_num_max_group=100, max_per_s1=120, nofill_max_group=50))
S1_PARAMS = MatcherParams(backend="xgb", device="cuda", num_leaves=127, learning_rate=0.1,
                          n_estimators=3000, early_stopping=50)
N_STAGE1 = 110_000  # entities per country (~213k, ~15M pairs); the bags use every row
S1_BAGS = 2
S2_PARAMS = MatcherParams(backend="xgb", device="cuda", n_estimators=4000, num_leaves=127, learning_rate=0.05, min_child_weight=1.0)
SEEDS = [42, 7, 2026]          # stage-2 seeds, averaged per cross-fitting part (SeedMean)
tcfg = TwoStageConfig(floor=0.01, max_cands=16, cohesion=False, rivals=True, model=S2_PARAMS,
                      train_roles=('fit', 'tune'),
                      extra_groups=('tok_evidence',))
v107 = json.loads((C.EXPERIMENTS / "v107_tight_rule" / "metrics.json").read_text())
parent = json.loads((C.EXPERIMENTS / "v110_m3_features" / "metrics.json").read_text())
EST_TO_BEAT = parent["metrics"]["est_public"]      # the parent's est_public is the bar
v103 = json.loads((C.EXPERIMENTS / "v103_mock_rule" / "metrics.json").read_text())
STAGE1_CACHE = (cfg.cache_dir / "stage1" /
                f"v120_{cfg.blocking.key()}_f{tcfg.floor}_k{tcfg.max_cands}_a{int(tcfg.anchors)}"
                f"_c{int(tcfg.cohesion)}_r{int(tcfg.rivals)}")
timings: dict[str, float] = {}
t_start = time.time()
print("blocking", cfg.blocking.key(), "|", len(cfg.feature_groups), "feature groups",
      "| v110 est_public to beat", round(EST_TO_BEAT, 4), "| normalise", cfg.normalise,
      "| stage 2", tcfg.record())

## 3. Data

The mock fold (as in v103–v107; its blocking under this configuration is cached or built on
first use) and the stage-1 training entities: train-fold S1 absent from the mock, capped at
`N_STAGE1` by id hash (`sample_s1`), about 9M candidate pairs.

In [ ]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
    # the transliteration token map learned from train-fold pairs (cached; identical to the
    # map of v101, so this notebook needs no earlier version's artifacts)
    token_map = learn_token_map(cfg, train)
    # (the map above is re-learned under this version's normalisation rules)
    # learned from the train fold's pairs like the token map (cached): the pool's filler words
    # and the word log-odds of one-word near-duplicates (fillers vs decoy markers)
    fillers = learn_fillers(cfg, train)
    evidence = learn_token_evidence(cfg, train)
print("fillers:", fillers, "| token evidence words (pool-only, S1-only):",
      None if evidence is None else (len(evidence.pool), len(evidence.s1)))
del val, fit_fold, tune_fold, fit_sample
absent = absent_from_mock(mock, train)
s1_ids = pd.Index(sample_s1(train.s1[isin(train.s1[C.ENTITY_ID], absent)], N_STAGE1)[C.ENTITY_ID])
print(f"absent from the mock: {len(absent):,}; stage-1 training entities: {len(s1_ids):,}")
train.s1[isin(train.s1[C.ENTITY_ID], s1_ids)][C.COUNTRY].value_counts()

## 4. Method

### 4.1 Stage 1 v2 on the GPU

Blocking of the training entities against the train-fold pool (cached), features chunk by
chunk to disk, then XGBoost on the GPU reading the chunks through a `DataIter`
(`stage1.fit_stage1`). The chunks are deleted once the model is saved.

In [ ]:
work = cfg.cache_dir / "stage1_chunks" / f"v120_{cfg.blocking.key()}"
t0 = time.time()
manifest = (json.loads((work / "manifest.json").read_text()) if (work / "manifest.json").exists()
            else write_chunks(cfg, train, s1_ids, token_map, work, timings=timings,
                              fillers=fillers, evidence=evidence))
timings["stage1_set_seconds"] = round(time.time() - t0, 2)
print(f"training set {timings['stage1_set_seconds']:.0f} s: {manifest['rows']:,} pairs of "
      f"{manifest['entities']:,} entities, positive rate {manifest['positives'] / manifest['rows']:.3f}")
t0 = time.time()
s1_model = fit_stage1(manifest, S1_PARAMS, bags=S1_BAGS)
timings["stage1_fit_seconds"] = round(time.time() - t0, 2)
stage1 = Fitted(s1_model, DecisionRule(), pd.DataFrame({"f_beta": [np.nan]}), cfg, token_map,
                {"stage1": "xgb gpu", "fit_info": s1_model.fit_info_}, fillers, evidence)
stage1.save(ARTIFACTS / "stage1_v2")
clean(work)
print(json.dumps(s1_model.fit_info_, indent=1))
s1_model.importance().head(15).rename("gain share").to_frame()

### 4.2 Stage 1 alone on the mock

The stage-1 pass keeps every pair with p1 ≥ 0.01 among its entity's 16 best: all pairs any
threshold rule could keep. The threshold rule on `p1`, tuned on the mock tune entities, gives
stage 1's single-stage mock F0.5, comparable with v103 (v101 + mock rule).

In [ ]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_mock_seconds"] = round(time.time() - t0, 2)
from entity_resolution.decision import one_to_one_filter
keep_ids = pd.Index(mock.fold.s1[C.ENTITY_ID][mock.role.isin(["tune", "val"]).to_numpy()])
p1_scored = []
for o in outs.values():
    s = o.pairs.assign(prob=o.X["p1"].to_numpy())
    s = one_to_one_filter(s)
    p1_scored.append(s[isin(s[C.S1_ID], keep_ids)])
p1_scored = pd.concat(p1_scored, ignore_index=True)
rule1, _ = tune_mock(p1_scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
single = mock_scores(p1_scored, mock, rule1)
del p1_scored
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
filter_report = pd.DataFrame({r: blocking_report(kept, mock.part(r)) for r in ("tune", "val")}).T
print(f"stage-1 pass {timings['stage1_mock_seconds']:.0f} s; rule {rule1}")
display(single[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision",
                "pair_recall"]])
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]]

### 4.3 Stage 2 on top, as in v104

Cross-fitted stage 2 on the mock's fit entities (GPU), stage-2 probabilities with the
pool-side 1-to-1 across every present entity, the rule tuned on the mock tune entities for
the tight metric (threshold or expected-F0.5 rule, whichever scores higher there).

In [ ]:
t0 = time.time()
parts = []
for seed in SEEDS:                    # one cross-fitted pair of models per seed
    m, fit_info = fit_stage2(outs, mock, replace(tcfg, model=replace(S2_PARAMS, seed=seed)))
    parts += m
per_fold = [parts[f::tcfg.folds] for f in range(tcfg.folds)]
models = [SeedMean(ms) if len(ms) > 1 else ms[0] for ms in per_fold]
scored, report = mock_scored(outs, models, mock, tcfg)
scored.to_parquet(ARTIFACTS / "mock_scored.parquet", index=False)
rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
tune_part = mock.part("tune")
rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                gammas=(0.7, 0.85, 1.0, 1.2, 1.5, 2.0),
                                misses=(0.0, 0.05, 0.1, 0.2, 0.4), fp_weight=FP_WEIGHT)
rule, table = ((rule_e, table_e) if table_e["f_beta"].max() > table_t["f_beta"].max()
               else (rule_t, table_t))
timings["stage2_seconds"] = round(time.time() - t0, 2)
res = mock_scores(scored, mock, rule)
print(f"stage 2 {timings['stage2_seconds']:.0f} s; rule {rule} (tight-tuned)")
importance = pd.concat([m.importance() for m in models], axis=1).mean(axis=1)
display(importance.sort_values(ascending=False).head(15).rename("gain share").to_frame())
res[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision", "pair_recall",
     "entities"]]

## 5. Evaluation

est_public of stage 1 alone and of the two-stage model on the mock val entities, against v103
and v107 (their public scores are the calibration points of the tight mock).

In [ ]:
cmp = pd.DataFrame({
    "v107 (public 0.966)": {"est_public": v107["metrics"]["comparison"]["v107 tight rule"]["est_public"],
                            "mock_f05": v107["mock_f05"]},
    "v110 (parent)": {"est_public": EST_TO_BEAT, "mock_f05": parent["mock_f05"]},
    "v120 stage 1 alone": {"est_public": single.loc["all", "est_public"],
                           "mock_f05": single.loc["all", "f_beta"]},
    "v120 two-stage": {"est_public": res.loc["all", "est_public"],
                       "mock_f05": res.loc["all", "f_beta"]},
}).T
cmp["delta_vs_parent"] = cmp["est_public"] - EST_TO_BEAT
for c in ("India", "US"):
    cmp.loc["v120 two-stage", f"est_public_{c}"] = res.loc[c, "est_public"]
cmp.round(4)

### Ablation

Stage 2 retrained (one seed) without the columns of ['tok_evidence']:
same parts, same early stopping, threshold rule tuned on the tight mock; the full model's
threshold-rule score is the like-for-like reference.

In [ ]:
from entity_resolution.features import FEATURE_COLUMNS
ABLATE = ('tok_evidence',)
cols_all = list(next(iter(outs.values())).X.columns)
drop = {c for g in ABLATE for c in FEATURE_COLUMNS[g]}
rows = {"full, threshold rule": mock_scores(scored, mock, rule_t).loc["all", ["f_beta", "est_public"]]}
t0 = time.time()
if drop:
    m_ab, _ = fit_stage2(outs, mock, tcfg, columns=[c for c in cols_all if c not in drop])
    s_ab, _ = mock_scored(outs, m_ab, mock, tcfg)
    r_ab, _ = tune_mock(s_ab, mock, cfg.grid, fp_weight=FP_WEIGHT)
    rows[f"without {list(ABLATE)}"] = mock_scores(s_ab, mock, r_ab).loc["all", ["f_beta", "est_public"]]
    del m_ab, s_ab
ablation = pd.DataFrame(rows).T
ablation["delta"] = ablation["est_public"] - ablation.loc["full, threshold rule", "est_public"]
print(f"ablation {time.time() - t0:.0f} s; {len(drop)} columns dropped of {len(cols_all)}")
ablation.round(4)

## 6. Error analysis

Error counts on the mock val entities against the parent's, then where each missed true pair
was lost: never a candidate (blocking or the stage-1 filter), taken by another S1 entity in
the pool-side 1-to-1, or below the rule.

In [ ]:
part = mock.part("val")
matches = apply_rule(scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))], rule)
counts = {k: len(error_samples(matches, part, k, n=10**9))
          for k in ("false_merge", "missed", "false_singleton", "singleton_merge")}
KEY = [C.S1_ID, C.ENTITY_ID]
truth = part.pairs[KEY]
country = truth[C.S1_ID].map(part.s1.set_index(C.ENTITY_ID)[C.COUNTRY]).to_numpy()
cands = pd.concat([o.pairs[KEY] for o in outs.values()], ignore_index=True)


def member(pairs: pd.DataFrame) -> np.ndarray:
    """Whether each true pair of the val part is among ``pairs``."""
    hit = pairs[KEY].drop_duplicates().assign(_hit=True)
    return truth.merge(hit, on=KEY, how="left")["_hit"].notna().to_numpy()


got, in_c, in_s = member(matches), member(cands), member(scored)
stage = np.select([~in_c, ~in_s, ~got], ["not a candidate", "1-to-1 conflict", "below the rule"],
                  "found")
losses = pd.crosstab(stage, country, normalize="columns")
losses["all"] = pd.Series(stage).value_counts(normalize=True)
print("v120:", counts, "\nv110:", parent["metrics"].get("errors_mock"))
losses.round(4)

## 7. Log the result

In [ ]:
ts = TwoStage(stage1, models, rule, tcfg, table,
              {"fit": fit_info, "stage1_fit": s1_model.fit_info_,
               "filter_report": filter_report.to_dict("index")})
ts.save(ARTIFACTS)
record = {
    "hypothesis": "every night-build switch on at once lifts the parent",
    "spec": {"version": "v120_night_build", "parent": "v110_m3_features", "stage1_groups": ["nofill"], "extra_groups": ["tok_evidence"], "blocking": {"nofill_max_group": 50}, "normalise": {"learn_fillers": true}, "evidence": {"learn": true}, "bags": 2, "stage2": {"train_roles": ["fit", "tune"], "num_leaves": 127, "learning_rate": 0.05, "min_child_weight": 1.0, "seeds": [42, 7, 2026]}, "ablate_groups": ["tok_evidence"], "notes": "Normalisation rules v5 (French N\u00b0 marker, compagnie, bis/crs/psg/appt, `+`/`et` -> and, leet legal forms, the record's own country word out of the core name) are the code default from this version on. M2's phonetic group was not rebased in time and is not in this build."},
    "feature_groups": list(cfg.feature_groups), "normalise": asdict(cfg.normalise),
    "seeds": SEEDS, "loss_breakdown": losses.to_dict(),
    "blocking_config": asdict(cfg.blocking), "stage1_params": asdict(S1_PARAMS),
    "stage1_cache": str(STAGE1_CACHE), "blocking_key": cfg.blocking.key(),
    "stage1_entities": len(s1_ids), "stage1_rows": manifest["rows"],
    "stage1_fit_info": s1_model.fit_info_, "single_stage_mock_f_beta": single.loc["all", "f_beta"],
    "single_stage_rule": asdict(rule1), "two_stage": tcfg.record(), "rule": asdict(rule),
    "rule_kind": type(rule).__name__, "fp_weight": FP_WEIGHT,
    "mock_f_beta": res.loc["all", "f_beta"], "est_public": res.loc["all", "est_public"],
    "f_tight": res.loc["all", "f_tight"], "single_stage_est_public": single.loc["all", "est_public"],
    **{f"mock_{c}": res.loc[c, "f_beta"] for c in ("India", "US")},
    **{k: res.loc["all", k] for k in ("f_beta_singletons", "f_beta_matched",
                                      "pair_precision", "pair_recall")},
    "cand_recall_val": filter_report.loc["val", "pair_recall"],
    "cands_mean_val": filter_report.loc["val", "candidates_mean"],
    "errors_mock": counts, "ablation": ablation.to_dict("index"), **timings,
    "peak_rss_gb": peak_rss_gb(),
}
DECISION = "KEEP" if record["est_public"] > EST_TO_BEAT + 0.001 else "DROP"
record["decision"] = DECISION
print(f"est_public v110 {EST_TO_BEAT:.4f} -> v120 {record['est_public']:.4f} {DECISION}")
row = log_result(
    EXP_DIR, change="night build: stage-1 groups +['nofill']; stage-2 extra groups ['tok_evidence']; blocking nofill_max_group=50; normalisation learn_fillers=True; token evidence learn=True; stage 1 in 2 bags; stage 2 on ['fit', 'tune'], 127 leaves, 3 seed(s)",
    group="INT", mock_f05=record["mock_f_beta"], cand_recall=record["cand_recall_val"],
    notes=(f"est_public {record['est_public']:.4f}; single stage est "
           f"{record['single_stage_est_public']:.4f}; cands {record['cands_mean_val']:.1f}/S1; "
           f"blocking {cfg.blocking.key()}"),
    metrics=record, owner="M1", parent="v110", decision=DECISION)
row

## 8. Conclusion

Written after the run from the numbers above.

## 9. Test inference

`run_test_two_stage` with this stage 1 and stage 2 (stage-1 outputs cached per stage 1,
blocking and filter), both validators, files kept in `submissions/v110/`.

In [ ]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, cache_dir=STAGE1_CACHE / "test")
print(f"run_test {time.time() - t0:.0f} s")
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
n_s1 = s1n_test.groupby(C.COUNTRY).size()
by = test_matches[C.S1_ID].map(country_of)
display(pd.DataFrame({
    "s1": n_s1,
    "cands_per_s1": test_summary["n_cands"].groupby(test_summary.index.map(country_of)).sum() / n_s1,
    "matched_share": test_matches.groupby(by)[C.S1_ID].nunique() / n_s1,
    "matches_per_s1": test_matches.groupby(by).size() / n_s1}))
dest = C.ROOT / "submissions" / "v120"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")